# # Load training data

In [1]:
# transformers is NLP-oriented — iris (tabular) doesn't apply.
# This notebook builds a tiny inline sentiment dataset so the template
# still demonstrates the train() context with a HuggingFace Trainer.
import pandas as pd
from nubison_model import data

raw = pd.DataFrame({
    "text": [
        "I love this product", "Amazing experience", "Wonderful day",
        "Best ever made", "Excellent quality", "Pretty good overall",
        "Terrible service", "I hate it", "Awful experience",
        "Worst day ever", "Disappointing result", "Not good at all",
    ],
    "target": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

datasets = data.split(raw, {"training": 0.75, "evaluation": 0.25}, random_state=42)
for name, sub in datasets.items():
    print(f"{name:12s} rows={len(sub):3d}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


training     rows=  9

evaluation   rows=  3

# # Train (HuggingFace transformers)

In [2]:
# Tokenize, fine-tune a tiny pretrained model, save weights — all
# inside the train() context. `report_to=["mlflow"]` tells the
# transformers Trainer to log to the active mlflow run.
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
)
from nubison_model import train

MODEL_ID = "distilbert-base-uncased"   # standard small transformer for the demo
WEIGHTS_PATH = "src/weights.pkl"


def _as_hf_dataset(X, y, tokenizer):
    df = X.assign(labels=y.values)
    ds = Dataset.from_pandas(df, preserve_index=False)
    return ds.map(
        lambda r: tokenizer(r["text"], padding="max_length", truncation=True, max_length=16),
        batched=True,
    )


with train(datasets=datasets, target="target", weights_path=WEIGHTS_PATH) as t:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

    train_ds = _as_hf_dataset(t.X, t.y, tokenizer)
    eval_ds = _as_hf_dataset(t.X_eval, t.y_eval, tokenizer)

    args = TrainingArguments(
        output_dir="/tmp/hf_trainer",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        eval_strategy="epoch",
        logging_steps=1,
        report_to=["mlflow"],          # log into the train() run
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds)
    trainer.train()

    t.save(model)

print(f"run_id: {t.run_id}")
print(f"weights: {WEIGHTS_PATH}")


2026/05/15 19:00:59 INFO mlflow.tracking.fluent: Experiment with name 'Transformers_1778839253' does not exist. Creating a new experiment.


2026/05/15 19:01:00 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


2026/05/15 19:01:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


2026/05/15 19:01:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


2026/05/15 19:01:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for transformers.


...

/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run bright-gnu-798 at: http://127.0.0.1:5050/#/experiments/13/runs/8da3e83b830b4fa2b06966f6f27346cb


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/13


run_id: 8da3e83b830b4fa2b06966f6f27346cb

weights: src/weights.pkl

# # Next: package for inference

The trained transformer is pickled at `src/weights.pkl`. To serve it as a
REST endpoint, see `model.ipynb` — note that the `infer` method should
tokenize the input text the same way and call the model accordingly.